In [ ]:
#| default_exp core

# API

> Configure Caddy through its admin API, with helpers for TLS and reverse-proxy routes

## How it works

fastcaddy reads and changes Caddy's JSON configuration through the [admin API](https://caddyserver.com/docs/api). It does not edit Caddyfiles. The default API address is `http://localhost:2019`.

You can configure Caddy at two levels:

- Read or change configuration by path with `gcfg`/`pcfg`, or by `@id` with `gid`/`pid`.
- Helpers such as `setup_caddy` and `add_reverse_proxy` configure TLS and routes. Each helper documents what it replaces or leaves unchanged.

Arrange how to restore the configuration after a Caddy restart. Our production apps run a setup script on each deploy. The script calls `reset()`, then adds the required configuration. `reset()` erases the entire configuration, including anything added outside the script.

Route helpers use these `@id` values to find existing routes:

- A reverse proxy uses its hostname.
- A wildcard route uses `wildcard-{domain}`.
- A subdomain proxy uses `{subdomain}.{domain}`.

Use `del_id` to remove a route by ID.

In [ ]:
import os, httpx, json
from fastcore.utils import *
from httpx import HTTPStatusError, ConnectError, get as xget, delete as xdelete
from typing import Sequence

In [ ]:
from fastcore.test import *

## Admin API primitives

`/config/` addresses configuration by path. `/id/` finds an object by its `@id` value anywhere in the configuration.

In [ ]:
admin_url = os.environ.get('CADDY_ADMIN', 'http://localhost:2019')

All admin-API access goes through a `CaddyAdmin`: it holds the target URL and an httpx client, and every primitive below is one of its methods, added with `@patch` next to the example that uses it. We build up the default instance method by method, then expose those methods as the module-level functions.

In [ ]:
class CaddyAdmin:
    "Drive one Caddy admin API. The module-level functions are bound methods of a default instance."
    def __init__(self,
        url=None, # Admin API base URL (defaults to the `admin_url` global, i.e. `$CADDY_ADMIN`)
        client=None): # httpx client (inject one with a mock transport for tests)
        self._url,self._client = url,client

    @property
    def url(self): return self._url or admin_url
    @property
    def client(self):
        if self._client is None: self._client = httpx.Client()
        return self._client

In [ ]:
default_admin = CaddyAdmin()

In [ ]:
@patch
def get_id(self:CaddyAdmin, path):
    "Get an ID full URL from a path"
    if path[0]!='/': path = '/'+path
    if path[-1]!='/': path = path+'/'
    return f'{self.url}/id{path}'

In [ ]:
default_admin.get_id('jph.answer.ai')

In [ ]:
@patch
def get_path(self:CaddyAdmin, path):
    "Get a config full URL from a path"
    if path[0]!='/': path = '/'+path
    if path[-1]!='/': path = path+'/'
    return f'{self.url}/config{path}'

In [ ]:
default_admin.get_path('/apps/tls/automation/policies')

Connection errors include the admin API address. HTTP errors include Caddy's error detail. `_req` adds these notes to the original exceptions before raising them.

In [ ]:
@patch
def _req(self:CaddyAdmin, method, url, **kw):
    "Request `url` via `method`, raising errors enriched with caddy's detail; returns the response"
    try: response = self.client.request(method, url, **kw)
    except ConnectError as e:
        e.add_note(f"Could not reach the caddy admin API at {self.url} -- is caddy running?")
        raise
    try: response.raise_for_status()
    except HTTPStatusError as e:
        try: msg = json.loads(response.text)['error']
        except Exception: msg = response.text
        if msg: e.add_note(f"Error: '{msg}'")
        raise
    return response

In [ ]:
@patch
def gid(self:CaddyAdmin, path='/'):
    "Get the config object whose `@id` matches `path`"
    return dict2obj(self._req('GET', self.get_id(path)).json())

In [ ]:
@patch
def has_id(self:CaddyAdmin, id):
    "Check if `id` is set up"
    try: self.gid(id)
    except HTTPStatusError: return False
    return True

In [ ]:
@patch
def gcfg(self:CaddyAdmin, path='/'):
    "Get the config at `path`"
    return dict2obj(self._req('GET', self.get_path(path)).json())

In [ ]:
@patch
def has_path(self:CaddyAdmin, path):
    "Check if any config exists at `path`"
    try: return self.gcfg(path) is not None
    except HTTPStatusError: return False

In [ ]:
@patch
def pid(self:CaddyAdmin, d, path='/', method='POST'):
    "Put/post config `d` to the object whose `@id` matches `path`"
    return self._req(method, self.get_id(path), json=obj2dict(d)).text or None

`method` selects a Caddy admin API operation:

- `post` sets a value, or appends to an array.
- `put` creates a value and any missing parent paths. An existing object key causes HTTP 409.
- `patch` replaces an existing value.
- `delete` removes a value.

Reading a missing key under an existing object returns `null`. Traversing through a missing key fails. `has_path` treats `None` as absence.

Posting to `/` replaces the entire configuration. `reset` posts an empty object. `nested_setcfg` also writes the entire document after modifying the requested path.

In [ ]:
@patch
def pcfg(self:CaddyAdmin, d, path='/', method='POST'):
    "Put/post config `d` at `path`"
    return self._req(method, self.get_path(path), json=obj2dict(d)).text or None

In [ ]:
@patch
def reset(self:CaddyAdmin):
    "Erase the entire caddy config"
    return self.pcfg({})

In [ ]:
@patch
def del_id(self:CaddyAdmin, id):
    "Delete every config object whose `@id` matches `id` (e.g. a host)"
    while self.has_id(id): self._req('DELETE', self.get_id(id))

With the methods in place, we expose them as module-level functions, each a bound method of `default_admin`. This is the pattern the standard library's `random` module uses: `random.random` and friends are bound methods of a hidden `Random()` instance. Existing code keeps calling `gid(...)`/`add_reverse_proxy(...)` unchanged, and because `default_admin.url` reads the `admin_url` global at call time, setting `fastcaddy.core.admin_url` still retargets them. Point a second Caddy, or a mock client in tests, with your own `CaddyAdmin(url=..., client=...)`.

In [ ]:
get_id,get_path,gid,has_id,gcfg,has_path,pid,pcfg,reset,del_id = (
    default_admin.get_id, default_admin.get_path, default_admin.gid, default_admin.has_id,
    default_admin.gcfg, default_admin.has_path, default_admin.pid, default_admin.pcfg,
    default_admin.reset, default_admin.del_id)

The admin API location comes from the `CADDY_ADMIN` env var if set. The functions below are bound methods of a default `CaddyAdmin` instance that reads the `admin_url` global at call time, so setting `fastcaddy.core.admin_url` still retargets them. For a second Caddy, a mock transport in tests, or a pooled client, construct your own `CaddyAdmin(url=..., client=...)` and call the same methods on it, leaving the module-level default untouched.


`CaddyAdmin` is what makes fastcaddy usable from a program rather than only a notebook. A second instance addresses a different Caddy without touching the global, and a mock-transport client exercises the whole API with no Caddy running (useful in tests):

In [ ]:
mock = CaddyAdmin(client=httpx.Client(
    transport=httpx.MockTransport(lambda r: httpx.Response(200, json={'ok': True}))))
test_eq(dict(mock.gcfg('/')), {'ok': True})
test_eq(reset.__self__, default_admin)


`reset` leaves an empty configuration.

In [ ]:
reset()
gcfg()

<div class="prose" markdown="1">

```python
{}
```

</div>

## Config tree helpers

In [ ]:
def nested_setdict(sd, value, *keys):
    "Returns `sd` updated to set `value` at the path `keys`"
    d = sd
    for key in keys[:-1]: d = d.setdefault(key, {})
    d[keys[-1]] = value
    return sd

In [ ]:
nested_setdict({'a':'b'}, {'c':'d'}, 'apps', 'http', 'servers', 'srv0')

{'a': 'b', 'apps': {'http': {'servers': {'srv0': {'c': 'd'}}}}}

In [ ]:
def path2keys(path):
    "Split `path` by '/' into a list"
    return path.strip('/').split('/')

In [ ]:
path2keys('/apps/tls/automation/policies')

['apps', 'tls', 'automation', 'policies']

In [ ]:
def keys2path(*keys):
    "Join `keys` into a '/' separated path"
    return '/'+'/'.join(keys)

In [ ]:
keys2path('apps', 'tls', 'automation', 'policies')

'/apps/tls/automation/policies'

In [ ]:
def nested_setcfg(value, *keys):
    "Set `value` at the path `keys` in the live caddy config"
    d = nested_setdict(gcfg(), value, *keys)
    return pcfg(d)

In [ ]:
def init_path(path):
    "Create `path` (and any missing parents) as an empty object, if not already present"
    if not has_path(path): pcfg({}, path, method='put')

`init_path` creates missing parents and leaves an existing path unchanged.

In [ ]:
init_path('/apps/tls/automation')
init_path('/apps/tls/automation')
test_eq(has_path('/apps/tls/automation'), True)
gcfg()

<div class="prose" markdown="1">

```python
{'apps': {'tls': {'automation': {}}}}
```

</div>

## TLS automation

Caddy's [automation policies](https://caddyserver.com/docs/json/apps/tls/automation/) control certificate issuance. Two helpers configure the first policy under `/apps/tls/automation`:

- `add_acme_config` uses ACME with Cloudflare DNS challenges.
- `add_tls_internal_config` uses Caddy's internal CA for local development.

Both leave existing automation configuration unchanged. Neither modifies configuration outside `/apps/tls/automation`.

In [ ]:
cf_token = os.environ.get('CADDY_CF_TOKEN', 'XXX')

In [ ]:
automation_path = '/apps/tls/automation'
def get_acme_config(token):
    "An ACME issuer config using cloudflare DNS challenges with `token`"
    prov = { "provider": { "name": "cloudflare", "api_token": token } }
    return { "module": "acme", "challenges": { "dns": prov } }

The issuer config that `add_acme_config` installs:

In [ ]:
get_acme_config('some-token')

{'module': 'acme',
 'challenges': {'dns': {'provider': {'name': 'cloudflare',
    'api_token': 'some-token'}}}}

In [ ]:
def add_tls_internal_config():
    "Set up a TLS automation policy using caddy's internal CA, if no automation config exists yet"
    if has_path(automation_path): return
    init_path(automation_path)
    pcfg([{"issuers": [{"module": "internal"}]}], automation_path+'/policies')

In [ ]:
def add_acme_config(cf_token, subjects=None):
    "Set up a TLS automation policy using ACME with cloudflare DNS challenges, if no automation config exists yet"
    if has_path(automation_path): return
    if not cf_token: raise ValueError("`cf_token` is required for ACME config")
    init_path(automation_path)
    policy = {'issuers': [get_acme_config(cf_token)]}
    if subjects: policy['subjects'] = subjects
    pcfg([policy], automation_path+'/policies')

Configure ACME with a valid Cloudflare API token:

In [ ]:
#| eval: false
add_acme_config(cf_token)

For local development, use the internal CA instead:

In [ ]:
reset()
add_tls_internal_config()
gcfg(automation_path)

<div class="prose" markdown="1">

```python
{'policies': [{'issuers': [{'module': 'internal'}]}]}
```

</div>

Check that `add_acme_config` leaves the internal CA policy unchanged. After a reset, it requires a token to create a new policy.

In [ ]:
add_acme_config(cf_token)
test_eq(gcfg(automation_path+'/policies/0/issuers/0/module'), 'internal')
reset()
test_fail(lambda: add_acme_config(None), contains='required')

## Schema reference

The package includes Caddy's JSON schema. `search_schema` finds matching keys and values. Pass a returned path to `get_schema` to read that node. Paths represent list indices as `[n]`.

In [ ]:
_caddy_docs = None

def caddy_docs():
    "The caddy JSON schema, loaded (once) from the bundled `caddy_schema.json`"
    global _caddy_docs
    if not _caddy_docs:
        pkg = Path(__file__).parent if '__file__' in globals() else Path('../fastcaddy')
        _caddy_docs = loads((pkg/'caddy_schema.json').read_text())
    return _caddy_docs

In [ ]:
def get_schema(path:str):
    "Get the caddy schema node at `path` (e.g. '/definitions/tls/properties/automation' or a path from `search_schema`)"
    node = caddy_docs()
    for part in path.strip('/').split('/'): node = node[int(part[1:-1])] if part.startswith('[') and part.endswith(']') else node[part]
    return node

In [ ]:
def _search_schema(term, d, path='', max_results=20):
    results = []
    if isinstance(d, dict):
        for k, v in d.items():
            p = f"{path}/{k}"
            if term.lower() in k.lower(): results.append(('key', p))
            results.extend(_search_schema(term, v, p, max_results))
    elif isinstance(d, list):
        for i, v in enumerate(d): results.extend(_search_schema(term, v, f"{path}/[{i}]", max_results))
    elif isinstance(d, str) and term.lower() in d.lower(): results.append(('value', path, d[:200]))
    return results[:max_results]

def search_schema(term:str, path:str='', max_results:int=20):
    "Recursively search the caddy schema for keys/values containing `term`; returned paths work with `get_schema`"
    d = get_schema(path) if path else caddy_docs()
    return _search_schema(term, d, path, max_results)

In [ ]:
search_schema('on_demand', max_results=3)

[('key', '/definitions/tls/properties/automation/properties/on_demand'),
 ('value',
  '/definitions/tls/properties/automation/properties/on_demand/description',
  'on_demand: object\nModule: tls\nhttps://pkg.go.dev/github.com/caddyserver/caddy/v2/modules/caddytls#OnDemandConfig\nOn-Demand TLS defers certificate operations to the\nmoment they are needed, e.g. during '),
 ('value',
  '/definitions/tls/properties/automation/properties/on_demand/markdownDescription',
  'on_demand: `object`  \nModule: `tls`  \n[godoc](https://pkg.go.dev/github.com/caddyserver/caddy/v2/modules/caddytls#OnDemandConfig)  \nOn-Demand TLS defers certificate operations to the\nmoment they are n')]

Check returned paths, including list indices, with `get_schema`.

In [ ]:
res = search_schema('issuers', max_results=50)
assert any('[' in p for _,p,*_ in res)
for _,p,*_ in res: get_schema(p)
get_schema(search_schema('on_demand')[0][1])['description']

"on_demand: object\nModule: tls\nhttps://pkg.go.dev/github.com/caddyserver/caddy/v2/modules/caddytls#OnDemandConfig\nOn-Demand TLS defers certificate operations to the\nmoment they are needed, e.g. during a TLS handshake.\nUseful when you don't know all the hostnames at\nconfig-time, or when you are not in control of the\ndomain names you are managing certificates for.\nIn 2015, Caddy became the first web server to\nimplement this experimental technology.\n\nNote that this field does not enable on-demand TLS;\nit only configures it for when it is used. To enable\nit, create an automation policy with `on_demand`.\n\n\nOnDemandConfig configures on-demand TLS, for obtaining\nneeded certificates at handshake-time. Because this\nfeature can easily be abused, Caddy must ask permission\nto your application whether a particular domain is allowed\nto have a certificate issued for it.\n"

## Routes and reverse proxies

The route helpers use an HTTP server named `srv0`. `init_routes` creates it on ports 80 and 443 when no servers exist. Routes go in `/apps/http/servers/srv0/routes`.

In [ ]:
srvs_path = '/apps/http/servers'
rts_path = srvs_path+'/srv0/routes'

In [ ]:
def init_routes():
    "Create the basic http server config (`srv0` on ports 80/443), if no servers exist yet"
    if has_path(srvs_path): return
    init_path(srvs_path)
    ir = dict(listen=[':80', ':443'], routes=[], protocols=['h1', 'h2'])
    pcfg(ir, f"{srvs_path}/srv0")

In [ ]:
def setup_pki_trust(install_trust):
    "Configure PKI certificate authority trust installation"
    if install_trust is None: return
    pki_path = '/apps/pki/certificate_authorities/local'
    init_path(pki_path)
    pcfg({"install_trust": install_trust}, pki_path)

In [ ]:
def setup_caddy(
    cf_token=None, # Cloudflare API token (required unless `local`)
    local:bool=False, # Use caddy's internal CA instead of ACME (for local dev)
    install_trust:bool=None, # Install the local CA into the system trust store?
    subjects=None # Subject names to restrict ACME cert issuance to
):
    "Create TLS automation config and the http server skeleton"
    if local: add_tls_internal_config()
    else: add_acme_config(cf_token, subjects=subjects)
    setup_pki_trust(install_trust)
    init_routes()

`setup_caddy` configures TLS automation and initializes `srv0`. It uses ACME by default or the internal CA with `local=True`. `install_trust` controls installation of the local CA in the system trust store.

In [ ]:
#| eval: false
setup_caddy(cf_token, subjects=['*.example.com', 'example.com'])

Use the internal CA for the remaining local examples:

In [ ]:
setup_caddy(local=True)
gcfg(srvs_path)

<div class="prose" markdown="1">

```python
{'srv0': {'listen': [':80', ':443'], 'protocols': ['h1', 'h2'], 'routes': []}}
```

</div>

A route's `handle` list contains its handlers. `encode_handler` builds response compression configuration. `proxy_handler` builds reverse-proxy configuration. You can combine them with other handlers in a custom route.

In [ ]:
def encode_handler():
    "An `encode` handler that compresses responses with zstd or gzip"
    enc = {"gzip": {"level": 1}, "zstd": {"level": "fastest"}}
    return dict(handler='encode', encodings=enc, prefer=['zstd', 'gzip'])

In [ ]:
def proxy_handler(
    *upstreams, # Upstream dial addresses, e.g. 'localhost:5001'
    st_delay='1m' # Keep streaming connections open this long across config reloads (None to disable)
):
    "A `reverse_proxy` handler dialing `upstreams`"
    res = {"handler": "reverse_proxy", "upstreams": [{"dial": u} for u in upstreams]}
    if st_delay: res["stream_close_delay"] = st_delay
    return res

In [ ]:
def add_route(route):
    "Append `route` to `srv0`'s route list"
    return pcfg(route, rts_path)

In [ ]:
def add_reverse_proxy(from_host, to_url, st_delay='1m', encode:bool=True):
    "Create (or replace) a route reverse-proxying `from_host` to `to_url`, tagged with `@id` `from_host`"
    if has_id(from_host): del_id(from_host)
    res = []
    if encode: res.append(encode_handler())
    res.append(proxy_handler(to_url, st_delay=st_delay))
    add_route({ "handle": res, "match": [{"host": [from_host]}], "@id": from_host, "terminal": True })

In [ ]:
host = 'foo.example.com'

In [ ]:
add_reverse_proxy(host, "localhost:5001")
gid(host)

<div class="prose" markdown="1">

```python
{ '@id': 'foo.example.com',
  'handle': [{'encodings': {'gzip': {'level': 1}, 'zstd': {'level': 'fastest'}}, 'handler': 'encode', 'prefer': ['zstd', 'gzip']}, {'handler': 'reverse_proxy', 'stream_close_delay': '1m', 'upstreams': [{'dial': 'localhost:5001'}]}],
  'match': [{'host': ['foo.example.com']}],
  'terminal': True}
```

</div>

Calling `add_reverse_proxy` again for the same host replaces its route without adding a duplicate.

In [ ]:
n = len(gcfg(rts_path))
add_reverse_proxy(host, "localhost:8000")
test_eq(len(gcfg(rts_path)), n)
test_eq(gid(host).handle[1].upstreams[0].dial, 'localhost:8000')

`del_id` removes every object with the ID, including duplicates created through `add_route`.

In [ ]:
add_route({"@id": host, "handle": [proxy_handler('localhost:9999')]})
del_id(host)
test_eq(has_id(host), False)

### On-demand TLS

Caddy obtains certificates during the TLS handshake for customers who point their own domains at your app.

The permission endpoint lets you prevent strangers from obtaining certificates through your server. Caddy sends it `GET {endpoint}?domain={domain}` before issuing a certificate. Return HTTP 200 to authorize the domain. `tools/testverify.py` provides a minimal FastHTML example.

`add_on_demand_tls` sets the permission endpoint. It appends an `on_demand: true` policy if none exists, preserving existing policies.

The appended policy has no `subjects` restriction. Caddy rejects a configuration with two unrestricted policies. Restrict earlier policies with `subjects`, as in this setup:

In [ ]:
def add_on_demand_tls(endpoint):
    "Enable on-demand TLS, asking `endpoint` for permission before each cert is issued"
    init_path(automation_path)
    od_path = automation_path+'/on_demand'
    od = {"permission": {"module": "http", "endpoint": endpoint}}
    pcfg(od, od_path, method='patch' if has_path(od_path) else 'put')
    policies_path = automation_path+'/policies'
    od_policy = {"on_demand": True, "issuers": [{"module": "acme"}]}
    if not has_path(policies_path): pcfg([od_policy], policies_path, method='put')
    else:
        policies = obj2dict(gcfg(policies_path))
        if not any(isinstance(o, dict) and o.get('on_demand') for o in policies):
            policies.append(od_policy)
            pcfg(policies, policies_path, method='patch')

In [ ]:
#| eval: false
setup_caddy(cf_token, subjects=["*.example.com", "example.com"])
add_on_demand_tls("http://localhost:5431/verifydom")

Use a subject-restricted internal CA policy for this local example. Check that the on-demand policy follows it and that a second call adds no duplicate.

In [ ]:
reset()
init_path(automation_path)
pcfg([{'subjects': ['*.example.com'], 'issuers': [{'module': 'internal'}]}], automation_path+'/policies')
add_on_demand_tls("http://localhost:5431/verifydom")
pols = gcfg(automation_path+'/policies')
test_eq(pols[1].on_demand, True)
add_on_demand_tls("http://localhost:5431/verifydom")
test_eq(len(gcfg(automation_path+'/policies')), 2)

## Wildcard subdomains

A wildcard route groups subdomains under a `*.{domain}` host matcher. Caddy can use one wildcard certificate for these hosts. Each subdomain has a route inside the wildcard's `subroute` handler.

Create the wildcard with `add_wildcard_route`, then add subdomains with `add_sub_reverse_proxy` or custom routes with `add_sub_route`. Calling `add_wildcard_route` again leaves the route and its subroutes unchanged.

In [ ]:
def add_wildcard_route(domain):
    "Add a route matching `*.{domain}` (tagged `wildcard-{domain}`) for subroutes; no-op if it already exists"
    wid = f"wildcard-{domain}"
    if has_id(wid): return
    add_route({"match": [{"host": [f"*.{domain}"]}], "handle": [{"handler": "subroute", "routes": []}], "@id": wid, "terminal": True})

In [ ]:
reset()
setup_caddy(local=True)
add_wildcard_route('something.example.com')
add_wildcard_route('something.example.com')
test_eq(sum(1 for o in gcfg(rts_path) if o['@id']=='wildcard-something.example.com'), 1)

`add_sub_route` appends a route inside the wildcard, replacing any existing route with its `@id`. Use it to configure custom handlers. Our production configuration combines `encode`, a custom router and `reverse_proxy` in one route.

In [ ]:
def add_sub_route(domain, route):
    "Append `route` to `domain`'s wildcard subroute list, replacing any existing route with the same `@id`"
    wid = f"wildcard-{domain}"
    if not has_id(wid): raise ValueError(f"No wildcard route for {domain} -- call `add_wildcard_route` first")
    rid = route.get('@id')
    if rid and has_id(rid): del_id(rid)
    pid([route], f"{wid}/handle/0/routes/...")

In [ ]:
def add_sub_reverse_proxy(
    domain, # Domain with an existing wildcard route
    subdomain, # Subdomain to proxy (tagged `{subdomain}.{domain}`)
    port:str|int|Sequence, # A single port or list of ports
    host='localhost', # Host the upstream(s) listen on
    st_delay='1m', # Keep streaming connections open this long across config reloads (None to disable)
    encode:bool=True # Compress responses?
):
    "Create (or replace) a reverse proxy to `{subdomain}.{domain}` inside `domain`'s wildcard route"
    route_id = f"{subdomain}.{domain}"
    if isinstance(port, (int,str)): port = [port]
    res = []
    if encode: res.append(encode_handler())
    res.append(proxy_handler(*[f"{host}:{p}" for p in port], st_delay=st_delay))
    add_sub_route(domain, {"@id": route_id, "match": [{"host": [route_id]}], "handle": res})

In [ ]:
add_sub_reverse_proxy('something.example.com', 'foo', 5001)
gid('foo.something.example.com')

<div class="prose" markdown="1">

```python
{ '@id': 'foo.something.example.com',
  'handle': [{'encodings': {'gzip': {'level': 1}, 'zstd': {'level': 'fastest'}}, 'handler': 'encode', 'prefer': ['zstd', 'gzip']}, {'handler': 'reverse_proxy', 'stream_close_delay': '1m', 'upstreams': [{'dial': 'localhost:5001'}]}],
  'match': [{'host': ['foo.something.example.com']}]}
```

</div>

Replace the subdomain's route with two upstream ports, then remove it with `del_id`.

In [ ]:
add_sub_reverse_proxy('something.example.com', 'foo', [5002, 5003])
subs = gid('wildcard-something.example.com').handle[0].routes
test_eq(len(subs), 1)
test_eq([u.dial for u in subs[0].handle[1].upstreams], ['localhost:5002', 'localhost:5003'])

In [ ]:
del_id('foo.something.example.com')
test_eq(has_id('foo.something.example.com'), False)

## Export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()